# Data Cleaning & Data-Quality Handling

> **Main explanation:** Raw data can contain quality issues. Data cleaning improves reliability and prepares the dataset for machine learning.

**Responsibilities covered:** missing-value analysis · duplicate-record analysis · invalid / inconsistent data handling · why cleaning was required · preprocessing decisions before modelling.

**Input:** the raw CSV (never modified). **Output:** `buildsafe_stage2_clean.csv` → used by Notebook 3.

> **About the outputs in this notebook.** Cells copied unchanged from the team's original notebooks keep the outputs from the original run on the real data.
> Cells that were newly written or reorganised for the viva have **no saved output** — run the notebook top-to-bottom (Kernel → Restart & Run All) before the evaluation so every number on screen is one you can explain.

## 1. Load the raw data and keep it untouched

**What:** load the raw CSV into `df_raw` and work on a copy `df`. **Why:** the raw file is the evidence of what we started with; every cleaning decision must be reversible and reproducible. **How:** `read_csv` then `.copy()`.

In [1]:
import os
import numpy as np
import pandas as pd

RAW_PATH = "../data/raw/DOB_ECB_Violations_20260913 (1).csv"
PROCESSED_DIR = "../data/processed"
os.makedirs(PROCESSED_DIR, exist_ok=True)

df_raw = pd.read_csv(RAW_PATH, low_memory=False)
print("Rows:", df_raw.shape[0])
print("Columns:", df_raw.shape[1])

Rows: 102467
Columns: 46


In [2]:
df = df_raw.copy()

print("Raw dataset:", df_raw.shape)
print("Working dataset:", df.shape)

Raw dataset: (102467, 46)
Working dataset: (102467, 46)


## 2. Missing-value analysis

**What:** count and rank missing values in every column. **Why:** most algorithms cannot accept missing values, and columns that are almost entirely empty add noise but no information. **How:** `isnull().sum()` and a per-column summary table.

In [3]:
missing = df.isnull().sum()

missing = missing[missing > 0].sort_values(ascending=False)

print(missing)

INFRACTION_CODE7             102467
SECTION_LAW_DESCRIPTION10    102467
INFRACTION_CODE10            102467
SECTION_LAW_DESCRIPTION9     102467
INFRACTION_CODE9             102467
SECTION_LAW_DESCRIPTION8     102467
INFRACTION_CODE8             102467
SECTION_LAW_DESCRIPTION7     102467
INFRACTION_CODE5             102467
SECTION_LAW_DESCRIPTION6     102467
INFRACTION_CODE6             102467
SECTION_LAW_DESCRIPTION5     102467
INFRACTION_CODE4             102467
SECTION_LAW_DESCRIPTION4     102467
INFRACTION_CODE3             102447
SECTION_LAW_DESCRIPTION3     102447
SECTION_LAW_DESCRIPTION2     100342
INFRACTION_CODE2              99864
DOB_VIOLATION_NUMBER          80813
CERTIFICATION_STATUS          31821
RESPONDENT_HOUSE_NUMBER       22754
RESPONDENT_ZIP                 8932
HEARING_STATUS                 8804
RESPONDENT_CITY                8279
SECTION_LAW_DESCRIPTION1       7994
RESPONDENT_STREET              7784
LOT                             487
BLOCK                       

In [4]:
summary = pd.DataFrame({
    "Column": df.columns,
    "Data_Type": df.dtypes.astype(str).values,
    "Missing_Count": df.isnull().sum().values,
    "Missing_Percentage": (df.isnull().mean() * 100).values,
    "Unique_Values": df.nunique(dropna=True).values
})

summary

,Column,Data_Type,Missing_Count,Missing_Percentage,Unique_Values
0,ISN_DOB_BIS_EXTRACT,int64,0,0.000000,102467
1,ECB_VIOLATION_NUMBER,str,0,0.000000,102467
2,ECB_VIOLATION_STATUS,str,0,0.000000,2
3,DOB_VIOLATION_NUMBER,str,80813,78.867343,19986
4,BIN,float64,95,0.092713,38743
5,BORO,int64,0,0.000000,5
6,BLOCK,float64,487,0.475275,9887
7,LOT,float64,487,0.475275,643
8,HEARING_DATE,int64,0,0.000000,568
9,HEARING_TIME,int64,0,0.000000,39


In [5]:
summary.sort_values(
    by="Missing_Percentage",
    ascending=False
)

,Column,Data_Type,Missing_Count,Missing_Percentage,Unique_Values
30,SECTION_LAW_DESCRIPTION4,float64,102467,100.000000,0
33,INFRACTION_CODE6,float64,102467,100.000000,0
34,SECTION_LAW_DESCRIPTION6,float64,102467,100.000000,0
31,INFRACTION_CODE5,float64,102467,100.000000,0
32,SECTION_LAW_DESCRIPTION5,float64,102467,100.000000,0
37,INFRACTION_CODE8,float64,102467,100.000000,0
38,SECTION_LAW_DESCRIPTION8,float64,102467,100.000000,0
39,INFRACTION_CODE9,float64,102467,100.000000,0
40,SECTION_LAW_DESCRIPTION9,float64,102467,100.000000,0
41,INFRACTION_CODE10,float64,102467,100.000000,0


In [ ]:
# Group the columns by how empty they are
pct = df.isnull().mean() * 100
bands = pd.cut(pct, bins=[-0.01, 0, 1, 10, 90, 99.999, 100],
               labels=["0 % (complete)", 
                       "0–1 %", 
                       "1–10 %", 
                       "10–90 %", 
                       "90–99.999 %", 
                       "100 % (empty)"])
print(bands.value_counts().sort_index())

print("\nColumns with partial missingness that we keep and must handle:")
partial = pct[(pct > 0) & (pct <= 10)].sort_values(ascending=False).round(2)
print(partial)

0 % (complete)    15
0–1 %              5
1–10 %             5
10–90 %            3
90–99.999 %        4
100 % (empty)     14
Name: count, dtype: int64

Columns with partial missingness that we keep and must handle:
RESPONDENT_ZIP              8.72
HEARING_STATUS              8.59
RESPONDENT_CITY             8.08
SECTION_LAW_DESCRIPTION1    7.80
RESPONDENT_STREET           7.60
BLOCK                       0.48
LOT                         0.48
BIN                         0.09
RESPONDENT_NAME             0.07
VIOLATION_DESCRIPTION       0.01
dtype: float64


**Decision framework**

| Missingness | Columns | Decision | Reason |
|---|---|---|---|
| 100 % or ≥ 97 % | `INFRACTION_CODE2–10`, `SECTION_LAW_DESCRIPTION2–10` | **drop** | no usable information; a model cannot learn from an empty column |
| 78.9 % | `DOB_VIOLATION_NUMBER` | **drop** (identifier) | it is an ID, not a feature |
| 0.01 – 8.7 % | `RESPONDENT_ZIP`, `RESPONDENT_CITY`, `SECTION_LAW_DESCRIPTION1`, `BLOCK`, `LOT`, `BIN`, `VIOLATION_DESCRIPTION` | **fill with `"UNKNOWN"`** | keeps every row; "missing" becomes its own category, which may itself be informative |

## 3. Duplicate-record analysis

**What:** test for exact duplicate rows and for repeated identifiers. **Why:** duplicates can bias training and, if copies land in both train and test sets, inflate scores. **How:** `df.duplicated()` and `is_unique` on identifiers.

In [7]:
print("Duplicate rows:", df.duplicated().sum())

Duplicate rows: 0


In [8]:
id_columns = [
    "ISN_DOB_BIS_EXTRACT",
    "ECB_VIOLATION_NUMBER",
    "DOB_VIOLATION_NUMBER",
    "BIN",
    "BLOCK",
    "LOT"
]

for col in id_columns:
    print("\n" + "=" * 60)
    print(col)
    print("Data type:", df[col].dtype)
    print("Total records:", len(df))
    print("Non-null:", df[col].notna().sum())
    print("Unique:", df[col].nunique(dropna=True))
    print("Missing:", df[col].isna().sum())


ISN_DOB_BIS_EXTRACT
Data type: int64
Total records: 102467
Non-null: 102467
Unique: 102467
Missing: 0

ECB_VIOLATION_NUMBER
Data type: str
Total records: 102467
Non-null: 102467
Unique: 102467
Missing: 0

DOB_VIOLATION_NUMBER
Data type: str
Total records: 102467
Non-null: 21654
Unique: 19986
Missing: 80813

BIN
Data type: float64
Total records: 102467
Non-null: 102372
Unique: 38743
Missing: 95

BLOCK
Data type: float64
Total records: 102467
Non-null: 101980
Unique: 9887
Missing: 487

LOT
Data type: float64
Total records: 102467
Non-null: 101980
Unique: 643
Missing: 487


In [9]:
print(
    "ECB_VIOLATION_NUMBER unique:",
    df["ECB_VIOLATION_NUMBER"].nunique()
)

print(
    "ECB_VIOLATION_NUMBER missing:",
    df["ECB_VIOLATION_NUMBER"].isna().sum()
)

print(
    "Is every ECB violation number unique?",
    df["ECB_VIOLATION_NUMBER"].is_unique
)

ECB_VIOLATION_NUMBER unique: 102467
ECB_VIOLATION_NUMBER missing: 0
Is every ECB violation number unique? True


**Observation.** The raw data has **0 exact duplicates** and every `ECB_VIOLATION_NUMBER` is unique — each row is a separate violation record. We re-check duplicates *after* removing the identifier columns in Section 8.

## 4. Invalid and inconsistent data

### 4.1 Dates: stored as integers, contain `0`, must be in a sensible order
**What:** check ranges, zero placeholders and chronology. **Why:** a `0` is not a real date, and a hearing that happens *before* the issue date would signal corrupt data. **How:** min/max, `== 0` counts, pairwise comparisons, then `pd.to_datetime(..., format="%Y%m%d", errors="coerce")`.

In [10]:
date_columns = [
    "ISSUE_DATE",
    "SERVED_DATE",
    "HEARING_DATE"
]

for col in date_columns:
    print("\n" + "=" * 60)
    print(col)

    print("Data type:", df[col].dtype)
    print("Missing:", df[col].isna().sum())
    print("Unique:", df[col].nunique())

    print("Minimum:", df[col].min())
    print("Maximum:", df[col].max())

    print("Most common:")
    print(df[col].value_counts(dropna=False).head(10))


ISSUE_DATE
Data type: int64
Missing: 0
Unique: 618
Minimum: 20250101
Maximum: 20260910
Most common:
ISSUE_DATE
20250408    985
20250225    956
20250701    861
20260420    726
20250129    565
20250211    453
20250528    424
20250108    404
20260302    350
20260730    343
Name: count, dtype: int64

SERVED_DATE
Data type: int64
Missing: 0
Unique: 619
Minimum: 0
Maximum: 20260910
Most common:
SERVED_DATE
20260604    396
20250424    375
20260507    371
20260521    371
20250807    370
20250501    370
20250717    368
20260601    367
20260512    365
20250731    362
Name: count, dtype: int64

HEARING_DATE
Data type: int64
Missing: 0
Unique: 568
Minimum: 20250206
Maximum: 20271222
Most common:
HEARING_DATE
20261007    815
20261016    756
20261110    755
20261014    658
20261002    648
20260807    643
20261030    617
20260715    616
20261104    615
20261028    593
Name: count, dtype: int64


In [11]:
for col in date_columns:
    print(
        col,
        "zero values:",
        (df[col] == 0).sum()
    )

ISSUE_DATE zero values: 0
SERVED_DATE zero values: 7
HEARING_DATE zero values: 0


In [12]:
print("SERVED before ISSUE:",
      (df["SERVED_DATE"] < df["ISSUE_DATE"]).sum())

print("HEARING before ISSUE:",
      (df["HEARING_DATE"] < df["ISSUE_DATE"]).sum())

print("HEARING before SERVED:",
      (df["HEARING_DATE"] < df["SERVED_DATE"]).sum())

SERVED before ISSUE: 7
HEARING before ISSUE: 0
HEARING before SERVED: 0


In [13]:
# Convert date columns to proper datetime format

date_columns = {
    "ISSUE_DATE": "ISSUE_DATE_DT",
    "SERVED_DATE": "SERVED_DATE_DT",
    "HEARING_DATE": "HEARING_DATE_DT"
}

for original_col, new_col in date_columns.items():
    df[new_col] = pd.to_datetime(
        df[original_col].astype(str),
        format="%Y%m%d",
        errors="coerce"
    )

print(df[
    [
        "ISSUE_DATE",
        "ISSUE_DATE_DT",
        "SERVED_DATE",
        "SERVED_DATE_DT",
        "HEARING_DATE",
        "HEARING_DATE_DT"
    ]
].head())

   ISSUE_DATE ISSUE_DATE_DT  SERVED_DATE SERVED_DATE_DT  HEARING_DATE  \
0    20260601    2026-06-01     20260601     2026-06-01      20260821   
1    20250805    2025-08-05     20250805     2025-08-05      20270415   
2    20260624    2026-06-24     20260624     2026-06-24      20260902   
3    20251203    2025-12-03     20251203     2025-12-03      20260908   
4    20260722    2026-07-22     20260819     2026-08-19      20261030   

  HEARING_DATE_DT  
0      2026-08-21  
1      2027-04-15  
2      2026-09-02  
3      2026-09-08  
4      2026-10-30  


In [14]:
for col in [
    "ISSUE_DATE_DT",
    "SERVED_DATE_DT",
    "HEARING_DATE_DT"
]:
    print(
        col,
        "| Missing:", df[col].isna().sum(),
        "| Min:", df[col].min(),
        "| Max:", df[col].max()
    )

ISSUE_DATE_DT | Missing: 0 | Min: 2025-01-01 00:00:00 | Max: 2026-09-10 00:00:00
SERVED_DATE_DT | Missing: 7 | Min: 2025-01-01 00:00:00 | Max: 2026-09-10 00:00:00
HEARING_DATE_DT | Missing: 0 | Min: 2025-02-06 00:00:00 | Max: 2027-12-22 00:00:00


In [15]:
# Validate converted date columns

dt_columns = [
    "ISSUE_DATE_DT",
    "SERVED_DATE_DT",
    "HEARING_DATE_DT"
]

for col in dt_columns:
    print("\n" + "=" * 50)
    print(col)
    print("Missing:", df[col].isna().sum())
    print("Minimum:", df[col].min())
    print("Maximum:", df[col].max())

print("\n" + "=" * 50)
print("CHRONOLOGICAL CHECKS")

print(
    "SERVED before ISSUE:",
    (df["SERVED_DATE_DT"] < df["ISSUE_DATE_DT"]).sum()
)

print(
    "HEARING before ISSUE:",
    (df["HEARING_DATE_DT"] < df["ISSUE_DATE_DT"]).sum()
)

print(
    "HEARING before SERVED:",
    (df["HEARING_DATE_DT"] < df["SERVED_DATE_DT"]).sum()
)


ISSUE_DATE_DT
Missing: 0
Minimum: 2025-01-01 00:00:00
Maximum: 2026-09-10 00:00:00

SERVED_DATE_DT
Missing: 7
Minimum: 2025-01-01 00:00:00
Maximum: 2026-09-10 00:00:00

HEARING_DATE_DT
Missing: 0
Minimum: 2025-02-06 00:00:00
Maximum: 2027-12-22 00:00:00

CHRONOLOGICAL CHECKS
SERVED before ISSUE: 0
HEARING before ISSUE: 0
HEARING before SERVED: 0


In [16]:
# Inspect records where SERVED_DATE was originally 0

missing_served = df[df["SERVED_DATE"] == 0]

print("Number of records:", len(missing_served))

print(
    missing_served[
        [
            "ECB_VIOLATION_NUMBER",
            "BIN",
            "ISSUE_DATE",
            "SERVED_DATE",
            "HEARING_DATE",
            "SEVERITY",
            "VIOLATION_TYPE",
            "ECB_VIOLATION_STATUS",
            "HEARING_STATUS"
        ]
    ].to_string(index=False)
)

Number of records: 7
ECB_VIOLATION_NUMBER       BIN  ISSUE_DATE  SERVED_DATE  HEARING_DATE  SEVERITY VIOLATION_TYPE ECB_VIOLATION_STATUS HEARING_STATUS
           39156479P 2055615.0    20250808            0      20260227 CLASS - 1      Elevators               ACTIVE   IN VIOLATION
           39149568L 3197976.0    20250602            0      20261008 CLASS - 1   Construction               ACTIVE        PENDING
           35483827P 2102335.0    20250216            0      20261113 CLASS - 1   Construction               ACTIVE        PENDING
           35482615Z 2077614.0    20250620            0      20261106 CLASS - 1   Construction               ACTIVE        PENDING
           39167302N 3399379.0    20251117            0      20260128 CLASS - 2   Construction               ACTIVE        DEFAULT
           35729351X 3025119.0    20260723            0      20261007 CLASS - 1   Construction               ACTIVE        PENDING
           35686012Y 3128689.0    20260319            0      2

**Decision.** `SERVED_DATE = 0` (7 rows) is a placeholder for "missing", so `errors="coerce"` turns it into `NaT`. After conversion there are **no chronology errors** (served/hearing never before issue). `SERVED_DATE` and `HEARING_DATE` are later dropped anyway because they happen *after* issue.

### 4.2 Money columns stored as text, with negative balances

In [17]:
financial_columns = ["PENALITY_IMPOSED", "AMOUNT_PAID", "BALANCE_DUE"]

In [18]:
for col in financial_columns:
    cleaned = (
        df[col]
        .astype(str)
        .str.replace(",", "", regex=False)
        .str.strip()
    )

    numeric = pd.to_numeric(cleaned, errors="coerce")

    print("\n" + "=" * 60)
    print(col)
    print("Could convert to numeric:", numeric.notna().sum())
    print("Could NOT convert:", numeric.isna().sum())

    if numeric.notna().any():
        print("Minimum:", numeric.min())
        print("Maximum:", numeric.max())


PENALITY_IMPOSED
Could convert to numeric: 102467
Could NOT convert: 0
Minimum: 0
Maximum: 70000

AMOUNT_PAID
Could convert to numeric: 102467
Could NOT convert: 0
Minimum: 0
Maximum: 57500

BALANCE_DUE
Could convert to numeric: 102467
Could NOT convert: 0
Minimum: -35000
Maximum: 70000


In [19]:
negative_balance = df[
    pd.to_numeric(
        df["BALANCE_DUE"].astype(str).str.replace(",", "", regex=False),
        errors="coerce"
    ) < 0
]

print("Negative BALANCE_DUE records:", len(negative_balance))

print(
    negative_balance[
        [
            "BALANCE_DUE",
            "AMOUNT_PAID",
            "PENALITY_IMPOSED",
            "SEVERITY",
            "ECB_VIOLATION_STATUS"
        ]
    ].head(20)
)

Negative BALANCE_DUE records: 408
     BALANCE_DUE AMOUNT_PAID PENALITY_IMPOSED   SEVERITY ECB_VIOLATION_STATUS
455          -60         680              620  CLASS - 2               ACTIVE
666       -1,280       2,560            1,280  CLASS - 2               ACTIVE
909         -620         620                0  CLASS - 2              RESOLVE
997       -1,250       1,250                0  CLASS - 2              RESOLVE
1120      -3,125       6,604            3,125  CLASS - 2              RESOLVE
1131        -120         120                0  CLASS - 3              RESOLVE
1393      -1,250       1,250                0  CLASS - 2              RESOLVE
1501        -620         620                0  CLASS - 2              RESOLVE
1537      -5,000      10,165            5,000  CLASS - 2              RESOLVE
1880      -3,125       1,280            3,125  CLASS - 2              RESOLVE
2650        -620         620                0  CLASS - 2              RESOLVE
2716        -345       1,000  

**Observation.** The three money columns are text such as `"1,250"`; they convert cleanly once commas are removed. `BALANCE_DUE` has **408 negative values** (over-payments). These columns describe the *outcome* of a case (penalty, payment, balance) and do not exist when a violation is first issued, so they are **excluded** as leakage rather than repaired — but identifying the problem shows we inspected them.

### 4.3 Infraction codes damaged by scientific notation

In [ ]:
sci = df["INFRACTION_CODE1"].astype(str).str.match(
    r"^\d+(\.\d+)?E[+-]?\d+$", 
    na=False)
print("INFRACTION_CODE1 values that look like scientific notation:", int(sci.sum()))
print(df.loc[sci, "INFRACTION_CODE1"].value_counts().head(10))

INFRACTION_CODE1 values that look like scientific notation: 1009
INFRACTION_CODE1
1.00E+08    614
1.00E+04    226
1.00E+05     92
1.00E+07     24
2.00E+02     19
2.00E+07      6
1.00E+09      6
1.00E+06      5
2.00E+04      4
1.00E+02      4
Name: count, dtype: int64


Codes such as `1.00E+01` look like codes that a spreadsheet converted to numbers (for example `1E1`). They are few, and one-hot encoding treats each as its own category, so they do no harm — but this is a **known data-quality defect** and should be repaired at source if a cleaner export is available.

### 4.4 Inconsistent respondent names

In [21]:
respondent_columns = [
    "RESPONDENT_NAME",
    "RESPONDENT_HOUSE_NUMBER",
    "RESPONDENT_STREET",
    "RESPONDENT_CITY",
    "RESPONDENT_ZIP"
]

for col in respondent_columns:
    print("\n" + "=" * 60)
    print(col)
    print("Data type:", df[col].dtype)
    print("Missing:", df[col].isna().sum())
    print("Missing %:", round(df[col].isna().mean() * 100, 2))
    print("Unique:", df[col].nunique())

    print("\nTop 10 values:")
    print(df[col].value_counts(dropna=False).head(10))


RESPONDENT_NAME
Data type: str
Missing: 75
Missing %: 0.07
Unique: 36932

Top 10 values:
RESPONDENT_NAME
DEPARTMENT OF EDUCATION      4264
DEPT OF EDUCATION            2869
DEPARTMENT OF EDUCATION 0     286
ON STAR MANAGEMENT LLC        235
INNOVATIVE CONSTR NYC LLC     227
L&L CONSTRUCTION DEVELOP      210
MOD CONSULTANTS LLC           197
YAFE MEOD CORP                167
NYC HOUSING AUTHORITY         161
MJH CONSTRUCTION CORP         158
Name: count, dtype: int64

RESPONDENT_HOUSE_NUMBER
Data type: str
Missing: 22754
Missing %: 22.21
Unique: 8702

Top 10 values:
RESPONDENT_HOUSE_NUMBER
NaN       22754
172-14      692
670         555
199         434
4118        423
40          395
1           382
130         346
253         313
250         262
Name: count, dtype: int64

RESPONDENT_STREET
Data type: str
Missing: 7784
Missing %: 7.6
Unique: 10036

Top 10 values:
RESPONDENT_STREET
NaN               7784
BROADWAY          1998
LEE AVENUE         735
5 AVENUE           733
89 AVENUE     

**Observation.** The same organisation appears under several spellings (`DEPARTMENT OF EDUCATION`, `DEPT OF EDUCATION`, `DEPARTMENT OF EDUCATION 0`). Respondent name, house number and street have tens of thousands of distinct values, so they are **dropped** instead of cleaned (also a privacy benefit). `VIOLATION_TYPE` contains a literal `"Unknown"` (13,585 rows) which is kept as a category.

## 5. Remove columns that cannot be used

**What:** drop identifiers, empty columns, post-issue columns, outcome-money columns and high-cardinality respondent fields. **Why:** they carry no information, add noise, or would leak information from after the violation was issued. **How:** `df.drop(columns=...)` with an existence check.

| Group | Columns | Why removed |
|---|---|---|
| Identifiers (3) | `ISN_DOB_BIS_EXTRACT`, `ECB_VIOLATION_NUMBER`, `DOB_VIOLATION_NUMBER` | unique per row / mostly empty, no predictive meaning |
| Empty / near-empty (18) | `INFRACTION_CODE2–10`, `SECTION_LAW_DESCRIPTION2–10` | 97 – 100 % missing |
| Post-issue events (8) | `SERVED_DATE(_DT)`, `HEARING_DATE(_DT)`, `HEARING_TIME`, `ECB_VIOLATION_STATUS`, `HEARING_STATUS`, `CERTIFICATION_STATUS` | happen after the prediction point → leakage |
| Case-outcome money (3) | `PENALITY_IMPOSED`, `AMOUNT_PAID`, `BALANCE_DUE` | outcomes of the case → leakage |
| High-cardinality respondent (3) | `RESPONDENT_NAME`, `RESPONDENT_HOUSE_NUMBER`, `RESPONDENT_STREET` | tens of thousands of values, inconsistent, privacy-sensitive |

In [22]:
identifier_columns = ["ISN_DOB_BIS_EXTRACT", "ECB_VIOLATION_NUMBER", "DOB_VIOLATION_NUMBER"]

sparse_columns = ([f"INFRACTION_CODE{i}" for i in range(2, 11)] +
                  [f"SECTION_LAW_DESCRIPTION{i}" for i in range(2, 11)])

post_issue_columns = ["SERVED_DATE", "SERVED_DATE_DT", "HEARING_DATE", "HEARING_DATE_DT",
                      "HEARING_TIME", "ECB_VIOLATION_STATUS", "HEARING_STATUS", "CERTIFICATION_STATUS"]

outcome_money_columns = ["PENALITY_IMPOSED", "AMOUNT_PAID", "BALANCE_DUE"]

high_cardinality_columns = ["RESPONDENT_NAME", "RESPONDENT_HOUSE_NUMBER", "RESPONDENT_STREET"]

groups = {
    "identifiers": identifier_columns,
    "empty / near-empty": sparse_columns,
    "post-issue events": post_issue_columns,
    "case-outcome money": outcome_money_columns,
    "high-cardinality respondent": high_cardinality_columns,
}

all_to_drop = [c for cols in groups.values() for c in cols]
not_found = [c for c in all_to_drop if c not in df.columns]
print("Columns not found (should be empty):", not_found)

for name, cols in groups.items():
    print(f"{name:30s} {len(cols):2d} columns")

df = df.drop(columns=all_to_drop)
print("\nShape after dropping:", df.shape)
print(list(df.columns))

Columns not found (should be empty): []
identifiers                     3 columns
empty / near-empty             18 columns
post-issue events               8 columns
case-outcome money              3 columns
high-cardinality respondent     3 columns

Shape after dropping: (102467, 14)
['BIN', 'BORO', 'BLOCK', 'LOT', 'ISSUE_DATE', 'SEVERITY', 'VIOLATION_TYPE', 'RESPONDENT_CITY', 'RESPONDENT_ZIP', 'VIOLATION_DESCRIPTION', 'INFRACTION_CODE1', 'SECTION_LAW_DESCRIPTION1', 'AGGRAVATED_LEVEL', 'ISSUE_DATE_DT']


## 6. Handle the remaining missing values

**What:** fill the remaining gaps in the columns we keep. **Why:** the encoders and models need a value in every cell. **How:** fill with the constant label `"UNKNOWN"`.

**Why a constant and not the mode/median?** For categorical / text fields a "missing" flag can itself be informative, and a constant does **not** learn anything from the data — so it cannot leak information from the test set. (If we had used a mean or median it would have to be computed on the training data only.) Rows are kept, because the missing share is small (≤ 8.7 %).

In [23]:
missing_remaining = (
    df.isnull()
      .sum()
      .to_frame("Missing_Count")
)

missing_remaining["Missing_Percentage"] = (
    missing_remaining["Missing_Count"] / len(df) * 100
)

missing_remaining = (
    missing_remaining[missing_remaining["Missing_Count"] > 0]
    .sort_values("Missing_Percentage", ascending=False)
)

print(missing_remaining)

                          Missing_Count  Missing_Percentage
RESPONDENT_ZIP                     8932            8.716953
RESPONDENT_CITY                    8279            8.079674
SECTION_LAW_DESCRIPTION1           7994            7.801536
BLOCK                               487            0.475275
LOT                                 487            0.475275
BIN                                  95            0.092713
VIOLATION_DESCRIPTION                 8            0.007807


In [24]:
# Handle missing categorical/text values

text_missing_columns = [
    "RESPONDENT_CITY",
    "RESPONDENT_ZIP",
    "SECTION_LAW_DESCRIPTION1",
    "VIOLATION_DESCRIPTION"
]

for col in text_missing_columns:
    df[col] = df[col].fillna("UNKNOWN")

print("Missing values after text handling:")

for col in text_missing_columns:
    print(
        f"{col}: {df[col].isna().sum()}"
    )

Missing values after text handling:
RESPONDENT_CITY: 0
RESPONDENT_ZIP: 0
SECTION_LAW_DESCRIPTION1: 0
VIOLATION_DESCRIPTION: 0


In [25]:
print("Dtypes:")
print(df[["BIN", "BLOCK", "LOT"]].dtypes)

print("\nMissing rows:")
print(df[["BIN", "BLOCK", "LOT"]].isna().sum())

print("\nRows where BIN is missing:")
print(
    df[df["BIN"].isna()][
        ["BIN", "BLOCK", "LOT", "BORO", "SEVERITY"]
    ].head(20)
)

Dtypes:
BIN      float64
BLOCK    float64
LOT      float64
dtype: object

Missing rows:
BIN       95
BLOCK    487
LOT      487
dtype: int64

Rows where BIN is missing:
      BIN  BLOCK  LOT  BORO   SEVERITY
5719  NaN    NaN  NaN     4  CLASS - 2
5735  NaN    NaN  NaN     3  CLASS - 2
5831  NaN    NaN  NaN     4  CLASS - 2
5866  NaN    NaN  NaN     2  CLASS - 2
5894  NaN    NaN  NaN     4  CLASS - 2
5900  NaN    NaN  NaN     3  CLASS - 1
5947  NaN    NaN  NaN     4  CLASS - 2
6009  NaN    NaN  NaN     4  CLASS - 2
6022  NaN    NaN  NaN     3  CLASS - 2
6123  NaN    NaN  NaN     3  CLASS - 2
6202  NaN    NaN  NaN     4  CLASS - 2
6220  NaN    NaN  NaN     5  CLASS - 1
6223  NaN    NaN  NaN     1  CLASS - 2
6235  NaN    NaN  NaN     5  CLASS - 2
6268  NaN    NaN  NaN     3  CLASS - 1
6295  NaN    NaN  NaN     4  CLASS - 2
6321  NaN    NaN  NaN     4  CLASS - 2
6364  NaN    NaN  NaN     2  CLASS - 2
6380  NaN    NaN  NaN     5  CLASS - 2
6416  NaN    NaN  NaN     4  CLASS - 2


In [26]:
# Handle missing building/location identifiers

building_columns = ["BIN", "BLOCK", "LOT"]

for col in building_columns:
    df[col] = df[col].astype("string").fillna("UNKNOWN")

print("Missing values after handling:")

print(df[building_columns].isna().sum())

print("\nSample values:")
print(df[building_columns].head(10))

print("\nDtypes:")
print(df[building_columns].dtypes)

Missing values after handling:
BIN      0
BLOCK    0
LOT      0
dtype: int64

Sample values:
         BIN   BLOCK    LOT
0  2026562.0  3813.0   19.0
1  1003388.0   279.0   69.0
2  5020516.0   795.0   41.0
3  4139680.0  6381.0   54.0
4  2016355.0  3277.0    2.0
5  3014376.0   817.0   46.0
6  3331428.0  8329.0  225.0
7  1063300.0  2135.0   60.0
8  5106881.0   565.0   46.0
9  3123714.0  5308.0    6.0

Dtypes:
BIN      string
BLOCK    string
LOT      string
dtype: object


**Observation.** After filling, no missing values remain in the kept columns. The 95 rows with no `BIN` have no identifiable building — they get the label `UNKNOWN`, and in Notebook 3 they are explicitly given **zero** building history.

## 7. Duplicate records after removing identifiers

**What:** check for duplicates again now that the unique IDs are gone. **Why:** two different violations can look identical on the columns we keep. **How:** `df.duplicated()`, then look the rows up in the raw data to see whether their original IDs differ.

In [ ]:
dup_extra = df.duplicated().sum()                      # extra copies
dup_mask = df.duplicated(keep=False)                   # every row that belongs to a duplicate group
print("Extra duplicate rows:", dup_extra)
print("Rows involved in a duplicate group:", dup_mask.sum())

# df kept the raw index, so we can look the same rows up in df_raw
ids = df_raw.loc[df.index[dup_mask], 
                 ["ISN_DOB_BIS_EXTRACT", 
                  "ECB_VIOLATION_NUMBER"]]

print("\nDistinct ECB violation numbers among them:", 
      ids["ECB_VIOLATION_NUMBER"].nunique())

print("Distinct ISN records among them:          ", 
      ids["ISN_DOB_BIS_EXTRACT"].nunique())

print("\nGroup-size distribution (how many identical copies per group):")
print(df[dup_mask].groupby(list(df.columns),
                           dropna=False).size().value_counts().sort_index())

Extra duplicate rows: 508
Rows involved in a duplicate group: 827

Distinct ECB violation numbers among them: 827
Distinct ISN records among them:           827

Group-size distribution (how many identical copies per group):
2     224
3      51
4      22
5       9
6       7
7       2
8       2
10      1
11      1
Name: count, dtype: int64


**What the team found in the original run:** 508 extra duplicate rows (827 rows involved in duplicate groups) — and **all 827 have different ECB and ISN identifiers**.

**Decision — keep them.** They are genuinely different violations that happen to look identical on the retained columns (e.g. several identical elevator violations issued to one building on one day). Deleting them would throw away real events, and it would understate how often such batches occur. Because train and test are split by **date** (Notebook 4), identical rows from one day can never sit on both sides of the split. *(The alternative — dropping them — is defensible; be ready to explain that either choice is justified as long as the reason is stated.)*

## 8. `AGGRAVATED_LEVEL` — excluded because its timing is unknown

**What:** decide whether to keep `AGGRAVATED_LEVEL`. **Why:** it is strongly linked to the target, which is exactly what makes a leakage suspect: an "aggravated offence" flag may be assigned after a case develops. **How:** compare its issue- and hearing-date ranges and its relationship with `SEVERITY`.

In [28]:
print("AGGRAVATED_LEVEL values:")
print(df_raw["AGGRAVATED_LEVEL"].value_counts(dropna=False))

print("\nAGGRAVATED_LEVEL by ISSUE_DATE:")
print(
    df_raw.groupby("AGGRAVATED_LEVEL")["ISSUE_DATE"]
    .agg(["min", "max", "count"])
)

print("\nAGGRAVATED_LEVEL by HEARING_DATE:")
print(
    df_raw.groupby("AGGRAVATED_LEVEL")["HEARING_DATE"]
    .agg(["min", "max", "count"])
)

AGGRAVATED_LEVEL values:
AGGRAVATED_LEVEL
NO                            94199
AGGRAVATED OFFENSE LEVEL 1     7323
AGGRAVATED OFFENSE LEVEL 2      945
Name: count, dtype: int64

AGGRAVATED_LEVEL by ISSUE_DATE:
                                 min       max  count
AGGRAVATED_LEVEL                                     
AGGRAVATED OFFENSE LEVEL 1  20250102  20260908   7323
AGGRAVATED OFFENSE LEVEL 2  20250102  20260902    945
NO                          20250101  20260910  94199

AGGRAVATED_LEVEL by HEARING_DATE:
                                 min       max  count
AGGRAVATED_LEVEL                                     
AGGRAVATED OFFENSE LEVEL 1  20250318  20271222   7323
AGGRAVATED OFFENSE LEVEL 2  20250318  20270610    945
NO                          20250206  20271015  94199


**Decision.** The data give no way to prove the flag exists at issue time (its issue-date and hearing-date ranges look similar for all three levels), and it relates strongly to the target (Level 2 = 82 % Class 1). When timing cannot be established safely, **exclude** it. The full leakage discussion is in Notebook 4.

In [29]:
model_df = df.drop(columns=["AGGRAVATED_LEVEL"]).copy()
print("Cleaned dataset shape:", model_df.shape)
print(list(model_df.columns))

Cleaned dataset shape: (102467, 13)
['BIN', 'BORO', 'BLOCK', 'LOT', 'ISSUE_DATE', 'SEVERITY', 'VIOLATION_TYPE', 'RESPONDENT_CITY', 'RESPONDENT_ZIP', 'VIOLATION_DESCRIPTION', 'INFRACTION_CODE1', 'SECTION_LAW_DESCRIPTION1', 'ISSUE_DATE_DT']


## 9. Final quality audit and save

**What:** verify the cleaned dataset and save it. **Why:** we must prove the cleaning worked before passing data on. **How:** shape, missing values, target distribution unchanged, save to CSV.

In [30]:
print("Raw shape:    ", df_raw.shape)
print("Cleaned shape:", model_df.shape)
print("Rows lost:    ", len(df_raw) - len(model_df))

remaining_missing = model_df.isnull().sum()
print("\nColumns with missing values:", int((remaining_missing > 0).sum()))

print("\nTarget distribution (%) — must equal the raw distribution:")
print(pd.DataFrame({
    "raw": df_raw["SEVERITY"].value_counts(normalize=True).mul(100).round(2),
    "cleaned": model_df["SEVERITY"].value_counts(normalize=True).mul(100).round(2),
}))

OUT_PATH = f"{PROCESSED_DIR}/buildsafe_stage2_clean.csv"
model_df.to_csv(OUT_PATH, index=False)
print("\nSaved:", OUT_PATH)

Raw shape:     (102467, 46)
Cleaned shape: (102467, 13)
Rows lost:     0

Columns with missing values: 0

Target distribution (%) — must equal the raw distribution:
             raw  cleaned
SEVERITY                 
CLASS - 2  56.22    56.22
CLASS - 1  39.69    39.69
CLASS - 3   4.09     4.09

Saved: ../data/processed/buildsafe_stage2_clean.csv
